In [1]:
# 셀 1: 설치 + 전체 import
!pip install -q transformers datasets accelerate evaluate torchaudio

import os, json, pickle, shutil
import numpy as np
import torch
import torchaudio
from pathlib import Path
from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from transformers import ASTForAudioClassification, ASTFeatureExtractor, TrainingArguments, Trainer
import evaluate

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00
GPU: Tesla T4
Memory: 15.6 GB


In [2]:
# 셀 2: FSLD 데이터 경로 찾기
DATASET_BASE = None
for candidate in [
    '/kaggle/input/soundtag-fsld-genres/fsld_genre_data',
    '/kaggle/input/soundtag-fsld-genres',
    '/kaggle/input/datasets/zoebae/soundtag-fsld-genres/fsld_genre_data',
    '/kaggle/input/datasets/zoebae/soundtag-fsld-genres',
]:
    if os.path.exists(candidate):
        subdirs = [d for d in os.listdir(candidate) if os.path.isdir(os.path.join(candidate, d))]
        if len(subdirs) >= 5:
            DATASET_BASE = candidate
            break

if not DATASET_BASE:
    print('❌ FSLD data not found! Searching...')
    !find /kaggle/input/ -name '*.wav' | head -3
else:
    genres = sorted([d for d in os.listdir(DATASET_BASE) if os.path.isdir(os.path.join(DATASET_BASE, d))])
    total = 0
    for g in genres:
        count = len(list(Path(DATASET_BASE, g).glob('*.wav')))
        total += count
        print(f'  {g}: {count}')
    print(f'\nTotal: {total} files, {len(genres)} genres')
    print(f'Path: {DATASET_BASE}')

  Ballad: 200
  Bedroom_Pop: 78
  Contemporary_RandB: 32
  Dance_Pop: 200
  Disco: 32
  EDM: 200
  Electropop: 200
  Funk_Pop: 78
  Hip_Hop: 200
  Hyperpop: 200
  Pop_Rock: 172
  Trap: 54

Total: 1646 files, 12 genres
Path: /kaggle/input/datasets/zoebae/soundtag-fsld-genres/fsld_genre_data


In [3]:
# 셀 3: 데이터셋 구성
class GenreDataset(torch.utils.data.Dataset):
    def __init__(self, file_paths, labels, feature_extractor, target_sr=16000):
        self.file_paths = file_paths
        self.labels = labels
        self.fe = feature_extractor
        self.sr = target_sr
    def __len__(self):
        return len(self.file_paths)
    def __getitem__(self, idx):
        try:
            waveform, sr = torchaudio.load(self.file_paths[idx])
            if waveform.shape[0] > 1:
                waveform = waveform.mean(dim=0, keepdim=True)
            waveform = waveform.squeeze(0)
            if sr != self.sr:
                waveform = torchaudio.transforms.Resample(sr, self.sr)(waveform)
            # 길이 맞추기 (10초 = 160000 샘플)
            max_len = self.sr * 10
            if len(waveform) > max_len:
                waveform = waveform[:max_len]
            elif len(waveform) < max_len:
                waveform = torch.nn.functional.pad(waveform, (0, max_len - len(waveform)))
            inputs = self.fe(waveform.numpy(), sampling_rate=self.sr, return_tensors='pt')
            return {'input_values': inputs['input_values'].squeeze(0), 'labels': self.labels[idx]}
        except Exception as e:
            return {'input_values': torch.zeros(1024, 128), 'labels': self.labels[idx]}

file_paths, genre_labels = [], []
for genre_dir in sorted(Path(DATASET_BASE).iterdir()):
    if genre_dir.is_dir():
        genre_name = genre_dir.name.replace('_', ' ').replace('RandB', 'R&B')
        for wav in list(genre_dir.glob('*.wav')) + list(genre_dir.glob('*.mp3')):
            file_paths.append(str(wav))
            genre_labels.append(genre_name)

le = LabelEncoder()
encoded_labels = le.fit_transform(genre_labels)
train_paths, test_paths, train_labels, test_labels = train_test_split(
    file_paths, encoded_labels, test_size=0.2, random_state=42, stratify=encoded_labels)

fe = ASTFeatureExtractor.from_pretrained('MIT/ast-finetuned-audioset-10-10-0.4593')
train_dataset = GenreDataset(train_paths, train_labels, fe)
test_dataset = GenreDataset(test_paths, test_labels, fe)

print(f'Train: {len(train_paths)}, Test: {len(test_paths)}, Classes: {len(le.classes_)}')
print(f'Genres: {list(le.classes_)}')

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Train: 1316, Test: 330, Classes: 12
Genres: [np.str_('Ballad'), np.str_('Bedroom Pop'), np.str_('Contemporary R&B'), np.str_('Dance Pop'), np.str_('Disco'), np.str_('EDM'), np.str_('Electropop'), np.str_('Funk Pop'), np.str_('Hip Hop'), np.str_('Hyperpop'), np.str_('Pop Rock'), np.str_('Trap')]


In [4]:
# 셀 4: AST 모델 + 학습
model = ASTForAudioClassification.from_pretrained(
    'MIT/ast-finetuned-audioset-10-10-0.4593',
    num_labels=len(le.classes_), ignore_mismatched_sizes=True)

accuracy_metric = evaluate.load('accuracy')
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)
    top3 = sum(labels[i] in np.argsort(logits[i])[-3:] for i in range(len(labels))) / len(labels)
    acc['top3_accuracy'] = top3
    return acc

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir='/kaggle/working/ast_fsld_model',
        eval_strategy='epoch', save_strategy='epoch',
        learning_rate=1e-5, per_device_train_batch_size=8,
        num_train_epochs=15, warmup_ratio=0.1, weight_decay=0.01,
        logging_steps=20, load_best_model_at_end=True,
        metric_for_best_model='accuracy', save_total_limit=2,
        fp16=True, dataloader_num_workers=2, report_to='none'),
    train_dataset=train_dataset, eval_dataset=test_dataset,
    compute_metrics=compute_metrics)

print('✅ Training...')
trainer.train()

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                        
------------------------+----------+----------------------------------------------------------------------------------------
classifier.dense.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527, 768]) vs model:torch.Size([12, 768])
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527]) vs model:torch.Size([12])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1245, training_loss=0.788346084916448, metrics={'train_runtime': 3656.4167, 'train_samples_per_second': 5.399, 'train_steps_per_second': 0.34, 'total_flos': 1.338152640357335e+18, 'train_loss': 0.788346084916448, 'epoch': 15.0})

In [5]:
# 셀 5: 평가 + 모델 저장 + K-pop 분석
# === 평가 ===
results = trainer.evaluate()
print(f"\n{'='*60}")
print(f"Accuracy: {results['eval_accuracy']:.1%}, Top-3: {results['eval_top3_accuracy']:.1%}")

preds = trainer.predict(test_dataset)
report = classification_report(preds.label_ids, np.argmax(preds.predictions, axis=-1),
                               target_names=le.classes_, zero_division=0)
print(report)

# === 결과 텍스트 파일로 저장 (세션 끝나도 Output에 남음) ===
with open('/kaggle/working/fsld_results.txt', 'w') as f:
    f.write(f"Accuracy: {results['eval_accuracy']:.1%}\n")
    f.write(f"Top-3: {results['eval_top3_accuracy']:.1%}\n\n")
    f.write(report)

# === 모델 저장 ===
save_path = '/kaggle/working/ast_fsld_model_final'
trainer.save_model(save_path)
fe.save_pretrained(save_path)
with open(os.path.join(save_path, 'label_encoder.pkl'), 'wb') as f:
    pickle.dump(le, f)
shutil.make_archive('/kaggle/working/ast_fsld_model_final', 'zip', save_path)
print('💾 Model saved + zipped')

# === K-pop 분석 ===
KPOP_DIR = None
for candidate in [
    '/kaggle/input/soundtag-kpop-previews/deezer_data/previews',
    '/kaggle/input/datasets/zoebae/soundtag-kpop-previews/deezer_data/previews',
]:
    if os.path.exists(candidate):
        KPOP_DIR = candidate
        break

if KPOP_DIR:
    kpop_files = sorted(Path(KPOP_DIR).glob('*.mp3'))
    print(f'\nK-pop files: {len(kpop_files)}')
    model.eval()
    kpop_results = []
    for i, mp3 in enumerate(kpop_files):
        try:
            waveform, sr = torchaudio.load(str(mp3))
            if waveform.shape[0] > 1: waveform = waveform.mean(dim=0, keepdim=True)
            waveform = waveform.squeeze(0)
            if sr != 16000: waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)
            max_len = 16000 * 10
            if len(waveform) > max_len: waveform = waveform[:max_len]
            elif len(waveform) < max_len: waveform = torch.nn.functional.pad(waveform, (0, max_len - len(waveform)))
            inputs = fe(waveform.numpy(), sampling_rate=16000, return_tensors='pt')
            inputs = {k: v.to(model.device) for k, v in inputs.items()}
            with torch.no_grad():
                probs = torch.softmax(model(**inputs).logits, dim=-1).cpu().numpy()[0]
            top5 = [(str(le.classes_[j]), float(probs[j])) for j in np.argsort(probs)[-5:][::-1]]
            kpop_results.append({'filename': mp3.name, 'genres': top5})
            if i < 20:
                print(f'  {mp3.name:45s} → {", ".join(f"{g}({s:.0%})" for g,s in top5[:3])}')
        except Exception as e:
            if i < 3: print(f'  Error: {e}')
        if (i+1) % 200 == 0: print(f'  [{i+1}/{len(kpop_files)}]')

    top1 = Counter(r['genres'][0][0] for r in kpop_results)
    kpop_dist = '\n📊 K-pop 장르 분포:\n'
    for g, c in top1.most_common():
        kpop_dist += f'  {g:25s} {c:4d} ({c/len(kpop_results)*100:5.1f}%)\n'
    print(kpop_dist)

    # K-pop 결과 저장
    with open('/kaggle/working/ast_fsld_kpop_analysis.json', 'w') as f:
        json.dump(kpop_results, f, ensure_ascii=False, indent=2)
    # 텍스트 결과도 저장
    with open('/kaggle/working/fsld_results.txt', 'a') as f:
        f.write(kpop_dist)
    print('💾 K-pop analysis saved!')
else:
    print('⚠️ K-pop data not found. Add soundtag-kpop-previews dataset.')

print('\n✅ ALL DONE! Check Output tab for files.')


Accuracy: 45.2%, Top-3: 86.7%
                  precision    recall  f1-score   support

          Ballad       0.29      0.55      0.38        40
     Bedroom Pop       0.27      0.44      0.33        16
Contemporary R&B       0.14      0.17      0.15         6
       Dance Pop       0.43      0.65      0.51        40
           Disco       1.00      0.67      0.80         6
             EDM       0.48      0.30      0.37        40
      Electropop       0.70      0.78      0.74        40
        Funk Pop       0.44      0.25      0.32        16
         Hip Hop       0.19      0.12      0.15        40
        Hyperpop       0.33      0.07      0.12        40
        Pop Rock       0.76      0.74      0.75        35
            Trap       0.80      0.73      0.76        11

        accuracy                           0.45       330
       macro avg       0.49      0.46      0.45       330
    weighted avg       0.46      0.45      0.43       330



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

💾 Model saved + zipped

K-pop files: 1874
  ATEEZ_1021374282.mp3                          → Ballad(24%), Electropop(19%), Funk Pop(15%)
  ATEEZ_1021374292.mp3                          → Electropop(19%), Pop Rock(14%), Dance Pop(12%)
  ATEEZ_1021374302.mp3                          → Funk Pop(20%), Dance Pop(16%), Hip Hop(9%)
  ATEEZ_1021374312.mp3                          → Funk Pop(18%), Pop Rock(13%), Hip Hop(13%)
  ATEEZ_1021374322.mp3                          → EDM(16%), Disco(14%), Dance Pop(12%)
  ATEEZ_1021374332.mp3                          → Disco(16%), Pop Rock(14%), Electropop(12%)
  ATEEZ_1021374342.mp3                          → Ballad(45%), Dance Pop(10%), Funk Pop(7%)
  ATEEZ_2041936627.mp3                          → Ballad(27%), Electropop(26%), Pop Rock(17%)
  ATEEZ_2041936637.mp3                          → Electropop(14%), EDM(13%), Funk Pop(13%)
  ATEEZ_2041936647.mp3                          → EDM(16%), Dance Pop(14%), Funk Pop(14%)
  ATEEZ_2041936657.mp3            